# Four figure types from Kairui Hao's thesis

**Shows:** four separate figures on synthetic data (no inference is run).

1. `intervals`: 95% intervals of five parameters against a tuning parameter, over a baseline band.
2. `densities`: densities of three parameters, one line per method, with the true value marked.
3. `shaded_panels`: a grid of time series with a band from draws; two panels are shaded to set them apart.
4. `convergence`: an estimate and its spread settling over iterations, with a few chains on top.

**Graph types:** interval plot (`lf.intervals`), density plot (`lf.density`), band from draws (`lf.band`,
`lf.shade`), convergence trace (`lf.ribbon`, `lf.paths`).

**Writes:** `intervals.svg`, `densities.svg`, `shaded_panels.svg` and `convergence.svg` next to this notebook.

**To tweak:** change the capitalised settings at the top of a **Figure** cell and re-run that cell.
Every figure has its own **Data** cell just above it.

In [ ]:
import numpy as np

import labfigs as lf

lf.use()
rng = np.random.default_rng(2)

## Data for the intervals

Synthetic; swap this cell for your own loading.

- `beta`: the five x positions
- `interval_truths`: `{parameter name: true value}`, one panel each
- `interval_draws`: `{parameter name: draws}`, each of shape `(500, len(beta))`

In [ ]:
beta = 10.0 ** np.arange(1, 6)
interval_truths = {"a": 0.5, "b": 1.0, "c": 0.5, "d": 0.75, "e": 1.0}
interval_draws = {
    name: truth * (1 + 2 / beta + 0.4 * rng.normal(size=(500, 5)) / (beta / 10) ** 0.4)
    for name, truth in interval_truths.items()
}

## Figure 1: intervals

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `ROWS`, `COLUMNS`, `PANEL_ASPECT`: shape of the grid, and the height of one panel divided by its width
- `METHOD_LABEL`: legend entry of the intervals
- `BASELINE_LABEL`, `BASELINE_BAND`: legend entry of the grey band, and its edges as multiples of the true value
- `X_LABEL`: label of the x axis, which is logarithmic

The y label of each panel is built from the parameter's name, and its ticks sit at 0, 1 and 2 times the true value.

In [ ]:
NAME = "intervals"
ROWS, COLUMNS, PANEL_ASPECT = 2, 3, 0.7
METHOD_LABEL = "IFT"
BASELINE_LABEL, BASELINE_BAND = "NUTS", (0.9, 1.1)
X_LABEL = r"$\beta$"

fig, axes = lf.subplots(ROWS, COLUMNS, n=len(interval_truths), aspect=PANEL_ASPECT)
for ax, (name, truth) in zip(axes, interval_truths.items()):
    low, high = BASELINE_BAND
    ax.axhspan(low * truth, high * truth, color=lf.REFERENCE, alpha=0.3, lw=0, label=BASELINE_LABEL)
    lf.intervals(ax, beta, interval_draws[name], label=METHOD_LABEL)
    lf.truth(ax, truth)
    ax.set(xscale="log", xticks=beta, yticks=[0, truth, 2 * truth], xlabel=X_LABEL, ylabel=rf"${name}/\bar{{{name}}}$")
lf.legend_above(fig)
lf.save(fig, NAME)

## Data for the densities

Synthetic; swap this cell for your own loading.

- `methods`: names of the methods, which become legend entries
- `density_truths`: true value of each parameter, one column each
- `method_samples`: per column, `{method: samples}` with 400 samples each
- `filled_samples`: per column, the 400 samples of the filled density in the bottom row

In [ ]:
methods = ("Non-relaxed SVI", "Relaxed SVI", "Relaxed SGLD", "Relaxed PSGLD")
density_truths = (0.3, -1.0, 1.0)
method_samples, filled_samples = [], []
for truth in density_truths:
    method_samples.append({
        method: truth + 0.02 * rng.normal() + 0.01 * (m + 1) * rng.normal(size=400)
        for m, method in enumerate(methods)
    })
    filled_samples.append(truth + 0.03 * rng.normal(size=400))

## Figure 2: densities

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width
- `FILL_COLOUR`: colour of the filled density in the bottom row
- `Y_LABEL`: the one y label shared by all panels
- `LEGEND_COLUMNS`: number of columns in the legend above the figure

The x label of each column is built from its number.

In [ ]:
NAME = "densities"
PANEL_ASPECT = 0.55
FILL_COLOUR = "C1"
Y_LABEL = "Probability density"
LEGEND_COLUMNS = 3

fig, axes = lf.subplots(2, len(density_truths), aspect=PANEL_ASPECT)
for j, truth in enumerate(density_truths):
    for method, samples in method_samples[j].items():
        lf.density(axes[0, j], samples, label=method)
    lf.density(axes[1, j], filled_samples[j], fill=True, color=FILL_COLOUR)
    for ax in axes[:, j]:
        lf.truth(ax, truth, vertical=True)
        ax.set(xlabel=rf"$k_{j + 1}/\bar{{k}}_{j + 1}$", ylabel="")
fig.supylabel(Y_LABEL)
lf.legend_above(fig, ncols=LEGEND_COLUMNS)
lf.save(fig, NAME)

## Data for the shaded panels

Synthetic; swap this cell for your own loading.

- `hours`: time axis, length 200
- `measured_input`, `measured_output`: the two measured signals, same length
- `state_draws`: one entry per panel, each of shape `(draws, len(hours))`

In [ ]:
hours = np.linspace(0, 12, 200)
state = 32 * np.sin(2 * np.pi * hours / 12 + 0.4)
measured_input = 42 * np.sin(2 * np.pi * hours / 12 + 0.8)
measured_output = state + 1.5
state_draws = [state + (1 + i % 3) * rng.normal(size=(200, 1)) for i in range(6)]

## Figure 3: shaded panels

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `ROWS`, `COLUMNS`, `PANEL_ASPECT`: shape of the grid, and the height of one panel divided by its width
- `SHADED`: which panels get a grey background, counting from 0 along the rows
- `INPUT_LABEL`, `INPUT_COLOUR`: legend entry and colour of the measured input
- `STATE_LABEL`, `OUTPUT_LABEL`: legend entries of the band and of the measured output
- `LEFT_TITLE`, `RIGHT_TITLE`: titles at the two ends of each panel; `{}` becomes the panel number
- `X_LABEL`, `X_TICKS` and `Y_LABEL`, `Y_TICKS`: the shared axes

In [ ]:
NAME = "shaded_panels"
ROWS, COLUMNS, PANEL_ASPECT = 2, 3, 0.7
SHADED = [1, 5]
INPUT_LABEL, INPUT_COLOUR = "Measured input", "C1"
STATE_LABEL, OUTPUT_LABEL = "State posterior predictive", "Measured output"
LEFT_TITLE, RIGHT_TITLE = r"$T_{{\mathrm{{ex}},{}}}$", r"$T_{{\mathrm{{in}},{}}}$"
X_LABEL, X_TICKS = "$t$ (hr)", [0, 6, 12]
Y_LABEL, Y_TICKS = r"$T$ ($^\circ$C)", [-40, 0, 40]

fig, axes = lf.subplots(ROWS, COLUMNS, aspect=PANEL_ASPECT, sharex=True, sharey=True)
for i, ax in enumerate(axes.flat):
    ax.plot(hours, measured_input, color=INPUT_COLOUR, label=INPUT_LABEL)
    lf.band(ax, hours, state_draws[i], label=STATE_LABEL)
    ax.plot(hours, measured_output, color=lf.TRUE, ls="--", label=OUTPUT_LABEL)
    ax.set_title(LEFT_TITLE.format(i + 1), loc="left", color=INPUT_COLOUR)
    ax.set_title(RIGHT_TITLE.format(i + 1), loc="right", color="C0")
    ax.set(xticks=X_TICKS, yticks=Y_TICKS)
for ax in axes.flat[SHADED]:
    lf.shade(ax)
fig.supxlabel(X_LABEL)
fig.supylabel(Y_LABEL)
lf.legend_above(fig)
lf.save(fig, NAME)

## Data for the convergence traces

Synthetic; swap this cell for your own loading.

- `iterations`: x axis, length 300, in millions of iterations
- `chain_truths`: true value of each parameter, one panel each
- `estimates`: per panel, the estimate at each iteration
- `spread`: half-width of the ribbon around the estimate at each iteration
- `chains`: per panel, sampler chains of shape `(5, len(iterations))`

In [ ]:
iterations = np.linspace(0, 2, 300)
chain_truths = (0.3, -1.0, 1.0)
estimates = [truth * (1 - np.exp(-iterations / 0.2)) for truth in chain_truths]
spread = 0.05 + 0.6 * np.exp(-iterations / 0.1)
chains = [estimate + 0.03 * rng.normal(size=(5, 300)).cumsum(1) * np.exp(-iterations) for estimate in estimates]

## Figure 4: convergence

Settings at the top of the cell:

- `NAME`: file name of the figure, without `.svg`
- `PANEL_ASPECT`: height of one panel divided by its width
- `ESTIMATE_LABEL`: legend entry of the line with its ribbon
- `CHAIN_LABEL`, `CHAIN_COLOUR`, `CHAIN_ALPHA`: legend entry, colour and opacity of the chains
- `X_LABEL`, `X_TICKS`: the shared x axis
- `Y_LIMITS`, `Y_TICKS`: the y axis of every panel; the y label is built from the panel number

In [ ]:
NAME = "convergence"
PANEL_ASPECT = 0.75
ESTIMATE_LABEL = "Relaxed SVI"
CHAIN_LABEL, CHAIN_COLOUR, CHAIN_ALPHA = "Relaxed SGLD", "C1", 0.7
X_LABEL, X_TICKS = r"Iterations ($\times 10^6$)", [0, 1, 2]
Y_LIMITS, Y_TICKS = (-1.3, 1.3), [-1, 0, 1]

fig, axes = lf.subplots(1, len(chain_truths), aspect=PANEL_ASPECT, sharex=True)
for j, (ax, truth) in enumerate(zip(axes, chain_truths)):
    lf.ribbon(ax, iterations, estimates[j] - spread, estimates[j], estimates[j] + spread, label=ESTIMATE_LABEL)
    lf.paths(ax, iterations, chains[j], color=CHAIN_COLOUR, alpha=CHAIN_ALPHA, label=CHAIN_LABEL)
    lf.truth(ax, truth)
    ax.set(xticks=X_TICKS, yticks=Y_TICKS, ylim=Y_LIMITS, ylabel=rf"$k_{j + 1}/\bar{{k}}_{j + 1}$")
fig.supxlabel(X_LABEL)
lf.legend_above(fig)
lf.save(fig, NAME)